# E-commerce Data Analysis

This notebook covers the data cleaning, statistical analysis, visualization, and predictive modeling for the scraped book data.

## Setup
Install necessary dependencies.

In [ ]:
!pip install -r requirements.txt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import ipywidgets as widgets
from IPython.display import display

%matplotlib inline

## 0. Data Extraction

In [ ]:
# Run the scraper to generate books_raw.csv
!python scraper.py

## 1. Data Cleaning & Preprocessing

In [ ]:
# Load data

df = pd.read_csv('books_raw.csv')
display(df.head())

In [ ]:
# Initial inspection
df.info()

In [ ]:
# Data Cleaning Functions

def clean_price(price_str):
    if isinstance(price_str, str):
        return float(price_str.replace('£', ''))
    return price_str

rating_map = {
    'One': 1,
    'Two': 2,
    'Three': 3,
    'Four': 4,
    'Five': 5
}

def clean_rating(rating_str):
    return rating_map.get(rating_str, 0)

def clean_availability(avail_str):
    return 'In stock' in str(avail_str)


In [ ]:
# Apply Cleaning
df['Price_Cleaned'] = df['Price'].apply(clean_price)
df['Rating_Num'] = df['Rating'].apply(clean_rating)
df['In_Stock'] = df['Availability'].apply(clean_availability)

# Check for nulls and duplicates
print(f"Missing values:\n{df.isnull().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")

# Drop duplicates if any
df = df.drop_duplicates()

# Derive Price Category
def categorize_price(price):
    if price < 20:
        return 'Budget'
    elif 20 <= price <= 40:
        return 'Mid-range'
    else:
        return 'Premium'

df['Price_Category'] = df['Price_Cleaned'].apply(categorize_price)

display(df.head())

In [ ]:
# Save cleaned data
df.to_csv('books_cleaned.csv', index=False)
print("Cleaned data saved to books_cleaned.csv")

## 2. Statistical Analysis

In [ ]:
# Descriptive Stats
print("Price Statistics:")
print(df['Price_Cleaned'].describe())

# Mode
print(f"Mode Price: {df['Price_Cleaned'].mode()[0]}")

# Range
print(f"Price Range: {df['Price_Cleaned'].max() - df['Price_Cleaned'].min()}")

# Group Statistics (Avg Price by Category - Top 5)
top_categories = df['Category'].value_counts().head(5).index
df_top5 = df[df['Category'].isin(top_categories)]

print("\nAverage Price by Top 5 Category:")
print(df_top5.groupby('Category')['Price_Cleaned'].mean())


In [ ]:
# Inferential Statistics

# Outlier Detection (IQR)
Q1 = df['Price_Cleaned'].quantile(0.25)
Q3 = df['Price_Cleaned'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['Price_Cleaned'] < (Q1 - 1.5 * IQR)) | (df['Price_Cleaned'] > (Q3 + 1.5 * IQR))]
print(f"Number of price outliers: {len(outliers)}")

# Correlation
corr, p_value = stats.pearsonr(df['Price_Cleaned'], df['Rating_Num'])
print(f"Pearson Correlation (Price vs Rating): {corr:.4f}, p-value: {p_value:.4f}")


In [ ]:
# Hypothesis Testing
# Fiction vs Non-Fiction
# Assuming we can classify categories. For now, let's use two prominent categories if available, 
# or just two random large categories to demonstrate the t-test.
# Let's check unique categories first.
print(df['Category'].unique())

## 3. Data Visualization (Plotly & Interactive Dashboard)

In [ ]:
# Histogram: Price Distribution
fig_hist = px.histogram(df, x='Price_Cleaned', nbins=20, title='Price Distribution',
                        labels={'Price_Cleaned': 'Price (£)'})
fig_hist.add_vline(x=df['Price_Cleaned'].mean(), line_dash="dash", line_color="red", annotation_text="Mean")
fig_hist.show()

In [ ]:
# Box Plot: Price by Category (Top 5)
fig_box = px.box(df_top5, x='Category', y='Price_Cleaned', title='Price Distribution by Category (Top 5)')
fig_box.show()

In [ ]:
# Scatter Plot: Price vs Rating
fig_scatter = px.scatter(df, x='Rating_Num', y='Price_Cleaned', trendline="ols", 
                         title='Price vs Rating', hover_data=['Title', 'Category'])
fig_scatter.show()

In [ ]:
# Interactive Dashboard

def update_plot(category):
    if category == 'All':
        filtered_df = df
    else:
        filtered_df = df[df['Category'] == category]
    
    fig = px.scatter(filtered_df, x='Rating_Num', y='Price_Cleaned', 
                     title=f'Price vs Rating for {category}', 
                     hover_data=['Title'])
    fig.show()

category_dropdown = widgets.Dropdown(
    options=['All'] + list(df['Category'].unique()),
    value='All',
    description='Category:',
)

widgets.interactive(update_plot, category=category_dropdown)

## 4. Predictive Analysis

In [ ]:
# Prepare Data
X = pd.get_dummies(df[['Rating_Num', 'Category']], drop_first=True)
y = df['Price_Cleaned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train Model
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_squared_error(y_test, y_pred, squared=False)

print(f"R2 Score: {r2:.4f}")
print(f"MAE: {mae:.4f}")